1/x in encoder1 25030.75, 289 degrees
1/x in encoder2 68336.18, 326 degrees

In [1]:
import subprocess
from datasets import load_dataset
import re
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

dataset = load_dataset("dair-ai/emotion", split="test")

/Users/tonyma/code/FHE-BERT-Tiny-Emotion/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import pandas as pd
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gokuls/BERT-tiny-emotion-intent")

texts = dataset['text']
token_length = [len(tokenizer.encode(entry['text'], truncation=False)) for entry in dataset]
labels = dataset['label']

df = pd.DataFrame({
    'text': texts,
    'token_length': token_length,
    'label': labels
})

# sort label
df_label = df.sort_values(by=['label', 'token_length'], ascending=[True, True]).reset_index(drop=True)

In [3]:
new_df2 = df_label.groupby('label').head(10).reset_index(drop=True)
print(len(new_df2))
print(new_df2[:21])

60
                            text  token_length  label
0          i feel so embarrassed             6      0
1             i do feel stressed             6      0
2                i feels so lame             6      0
3      i feel embarrassed enough             6      0
4          i stop feeling guilty             6      0
5         i feel depressed again             6      0
6              i feel soo lonely             6      0
7     im feeling depressed again             6      0
8           i just feel troubled             6      0
9              i feel less alone             6      0
10          im feeling energetic             5      1
11          i feel more creative             6      1
12        i feel really inspired             6      1
13         i feel more energetic             6      1
14             i feel any better             6      1
15       i would feel productive             6      1
16   im feeling hopeful relieved             6      1
17           i feel gorge

In [4]:
correct = []
error = []
output = []
execution_times = []

func_error = []
count = 0

def run_fhe_bert_tiny(sample):
    global correct, error, func_error, output, execution_times, count
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    cmd = [f"./FHE-BERT-Tiny", sentence]
    #cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct.append(count)
            print("sadness")
        else:
            error.append(count)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct.append(count)
            print("joy")
        else:
            error.append(count)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct.append(count)
            print("love")
        else:
            error.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct.append(count)
            print("anger")
        else:
            error.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct.append(count)
            print("fear")
        else:
            error.append(count)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct.append(count)
            print("surprise")
        else:
            error.append(count)
    else: # error
        func_error.append(count)
        
    print(seconds, "seconds")

In [6]:
from tqdm import tqdm

for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    run_fhe_bert_tiny(row)
    count+=1

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:32<2:29:31, 152.05s/it]

[-1.56408e+49, -1.43798e+49, 3.66619e+49, -2.72294e+49, -7.39546e+48, 7.39771e+49]
139.0 seconds
sentence:  i do feel stressed


  3%|▎         | 2/60 [05:05<2:27:40, 152.77s/it]

...

[5.46026e+49, -3.21101e+49, -3.73336e+49, 3.35141e+49, 2.30662e+48, -9.0706e+48]
189.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [2:40:41<03:12, 192.85s/it]

[-3.06856e+49, 1.84936e+49, -1.70854e+49, 8.01628e+49, -2.3587e+49, -3.34167e+49]
186.0 seconds
sentence:  i always feel very shocked by that me threatening


100%|██████████| 60/60 [2:44:10<00:00, 164.18s/it]

[2.74414e+49, 1.94127e+49, 1.6175e+49, -6.95649e+49, -4.21676e+49, -2.92415e+49]
197.0 seconds


In [7]:
len(correct)

19

In [8]:
len(error)

41

In [9]:
error

[0,
 2,
 4,
 6,
 8,
 11,
 12,
 13,
 14,
 15,
 18,
 19,
 21,
 22,
 23,
 24,
 25,
 27,
 28,
 29,
 30,
 31,
 33,
 35,
 36,
 38,
 39,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 49,
 51,
 53,
 54,
 57,
 58,
 59]

In [10]:
print(len(output))
print(len(execution_times))

60
60


In [11]:
import numpy as np
np.savetxt("output_labels3_60", output, delimiter=",")
np.savetxt("execution_time_labels3_60", output, delimiter=",")

Test again

In [12]:
import math

new_resamples = []
for i in range(len(output)):
    if i in error:
        if (int(math.floor(math.log10(abs(output[i][0])))) > 0) or (int(math.floor(math.log10(abs(output[i][1])))) > 0) or (int(math.floor(math.log10(abs(output[i][2])))) > 0) or (int(math.floor(math.log10(abs(output[i][3])))) > 0) or (int(math.floor(math.log10(abs(output[i][4])))) > 0) or (int(math.floor(math.log10(abs(output[i][5])))) > 0):
            print(f"{i}: {output[i]}")
            new_resamples.append(i)
            
print(len(new_resamples))

0: [-1.56408e+49, -1.43798e+49, 3.66619e+49, -2.72294e+49, -7.39546e+48, 7.39771e+49]
2: [3.00351e+49, 4.09401e+49, -6.25281e+49, -4.18088e+49, 3.36665e+48, -1.31403e+49]
4: [-5.87309e+48, 1.31122e+49, 7.34202e+49, 1.45522e+49, 4.96568e+49, 1.48787e+49]
6: [-1.34051e+49, 2.24225e+49, -4.44251e+49, 1.77851e+49, 6.90386e+49, -6.40033e+49]
8: [3.97999e+48, -3.51563e+49, 1.67617e+49, 1.77801e+49, -2.49451e+49, 9.59318e+48]
11: [6.94056, -10.2813, -9.42732, 9.87348, 6.85065, -8.09456]
12: [2.69175e+49, 2.2116e+49, 1.82286e+49, -2.09838e+49, 4.2427e+49, -2.07749e+49]
13: [6.95906e+49, -8.55696e+49, 2.98016e+49, -3.32544e+49, -5.75707e+49, -3.4077e+49]
14: [-1.00916e+49, 2.42702e+48, -1.29311e+49, 8.88328e+48, 4.03457e+49, 1.59684e+49]
15: [3.21188e+49, 9.07834e+48, -3.1699e+49, 2.66788e+48, -4.54752e+49, 3.30469e+49]
18: [-2.05648e+49, -1.65535e+49, 1.16682e+49, -1.12357e+50, 3.62389e+49, 2.82455e+49]
19: [-1.25429e+49, -3.77034e+49, -1.04035e+49, -5.78357e+49, 6.06075e+49, 1.46966e+49]
25: 

In [13]:
new_resamples

[0,
 2,
 4,
 6,
 8,
 11,
 12,
 13,
 14,
 15,
 18,
 19,
 25,
 27,
 30,
 31,
 33,
 35,
 36,
 38,
 39,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 49,
 51,
 53,
 54,
 57,
 58,
 59]

In [14]:
# check again
for i in new_resamples:
    if not i in error:
        print("not exist: ", i)
        #new_resamples.remove(i)

In [15]:
correct2 = []
error2 = []
output2 = []
execution_times2 = []

func_error2 = []
count2 = 0

def run_fhe_bert_tiny2(sample):
    global correct2, error2, func_error2, output2, execution_times2, count2
    #print(sample)
    
    sentence = sample.text
    print("sentence: ", sentence)
    
    #cmd = [f"./FHE-BERT-Tiny", sentence]
    cmd = ["./FHE-BERT-Tiny", sentence, "--verbose"]
    
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )
    
    # format
    result = result.stdout.strip()
    #print("raw result", result)
    
    # check [x, x, x, x, x, x]
    #pattern = r'\[(.*?)\]'
    pattern = r'plain_result: \[(.*?)\]'
    match = re.search(pattern, result)
    tmp = match.group(1).split(",")
    res = [float(x) for x in tmp]
    output2.append(res)
    print(res)
    
    # execution time
    # The evaluation of the FHE circuit took: 121 seconds.
    pattern_time = r'The evaluation of the FHE circuit took:\s*(\d+)\s*seconds'
    match_time = re.search(pattern_time, result)
    seconds = float(match_time.group(1))
    execution_times2.append(seconds)
        
    if ("Outcome: sadness" in result):
        if (sample.label == 0):
            correct2.append(count2)
            print("sadness")
        else:
            error2.append(count2)
    elif ("Outcome: joy" in result):
        if (sample.label == 1):
            correct2.append(count2)
            print("joy")
        else:
            error2.append(count2)
    elif ("Outcome: love" in result):
        if (sample.label == 2):
            correct2.append(count2)
            print("love")
        else:
            error2.append(count)
    elif ("Outcome: anger" in result):
        if (sample.label == 3):
            correct2.append(count2)
            print("anger")
        else:
            error2.append(count)
    elif ("Outcome: fear" in result):
        if (sample.label == 4):
            correct2.append(count2)
            print("fear")
        else:
            error2.append(count2)
    elif ("Outcome: surprise" in result):
        if (sample.label == 5):
            correct2.append(count2)
            print("surprise")
        else:
            error2.append(count2)
    else: # error
        error2.append(count2)
        
    print(seconds, "seconds")

In [16]:
from tqdm import tqdm

eliminated = 0
for row in tqdm(new_df2.itertuples(index=False), total=new_df2.shape[0]):
    if count2 == new_resamples[eliminated]:
        #print(count)
        eliminated+=1
        run_fhe_bert_tiny2(row)
    count2+=1
    
    if eliminated == len(new_resamples):
        break

  0%|          | 0/60 [00:00<?, ?it/s]

sentence:  i feel so embarrassed


  2%|▏         | 1/60 [02:27<2:24:50, 147.30s/it]

[5.111399422198004e+49, -3.5154107609726623e+49, 1.0089741578178623e+49, -2.1653270308449126e+49, -3.729187629964513e+49, -5.593905425552421e+49]
sadness
135.0 seconds
sentence:  i feels so lame


  5%|▌         | 3/60 [04:59<1:29:44, 94.46s/it] 

...

[-1.611176301171392e+49, -1.09149174216381e+49, -1.48170200798709e+49, -4.792211821858548e+49, -4.453848716264593e+49, -4.807485562287859e+49]
183.0 seconds
sentence:  i am feeling amazing and seeing the difference


 98%|█████████▊| 59/60 [1:35:34<01:50, 110.84s/it]

[-1.364089680107834e+49, -7.92319894301046e+48, 3.695440314293132e+49, 1.4046104776860718e+49, -3.000031408706494e+49, -7.169621107849976e+49]
184.0 seconds
sentence:  i always feel very shocked by that me threatening


 98%|█████████▊| 59/60 [1:39:03<01:40, 100.74s/it]

[1.9402881662320778e+49, 2.082473028147911e+48, -4.4005446077721113e+49, 5.841888595173765e+49, 9.017406507324457e+48, 4.4858959857920695e+49]
197.0 seconds


In [17]:
len(correct2)

12

In [18]:
# total accuracy
total_accuracy = correct + correct2
total_accuracy.sort()
print(len(total_accuracy))

31


In [19]:
len(total_accuracy)/60

0.5166666666666667

In [20]:
# Each label accuracy
label0_accuracy = []
label1_accuracy = []
label2_accuracy = []
label3_accuracy = []
label4_accuracy = []
label5_accuracy = []

for i in total_accuracy:
    if i < 10:
        label0_accuracy.append(i)
    elif i < 20:
        label1_accuracy.append(i)
    elif i < 30:
        label2_accuracy.append(i)
    elif i < 40:
        label3_accuracy.append(i)
    elif i < 50:
        label4_accuracy.append(i)
    elif i < 60:
        label5_accuracy.append(i)

In [21]:
print(len(label0_accuracy))
print(len(label1_accuracy))
print(len(label2_accuracy))
print(len(label3_accuracy))
print(len(label4_accuracy))
print(len(label5_accuracy))

8
6
2
7
3
5


In [22]:
print(df_label[df_label['label'] == 0].shape[0])
print(df_label[df_label['label'] == 1].shape[0])
print(df_label[df_label['label'] == 2].shape[0])
print(df_label[df_label['label'] == 3].shape[0])
print(df_label[df_label['label'] == 4].shape[0])
print(df_label[df_label['label'] == 5].shape[0])


581
695
159
275
224
66


In [23]:
new_resamples

[0,
 2,
 4,
 6,
 8,
 11,
 12,
 13,
 14,
 15,
 18,
 19,
 25,
 27,
 30,
 31,
 33,
 35,
 36,
 38,
 39,
 41,
 42,
 43,
 44,
 45,
 46,
 47,
 49,
 51,
 53,
 54,
 57,
 58,
 59]

In [24]:
# check final wrong answer
final_wrong = []

for i in range(60):
    if not i in total_accuracy and i in new_resamples:
        print(i)
        print(f"first: {output[i]}")
        print(f"retest: {output2[new_resamples.index(i)]}")
        final_wrong.append(i)

2
first: [3.00351e+49, 4.09401e+49, -6.25281e+49, -4.18088e+49, 3.36665e+48, -1.31403e+49]
retest: [-1.0154840661703794e+49, -2.8189915614764747e+49, 1.6096265287688839e+49, 1.7545215467858707e+49, -5.821601355440108e+48, -1.3555357414229281e+49]
4
first: [-5.87309e+48, 1.31122e+49, 7.34202e+49, 1.45522e+49, 4.96568e+49, 1.48787e+49]
retest: [-7.21945274534208e+48, -6.550544629698646e+49, -1.2825918980292213e+49, 8.156441459772905e+48, -3.2728389141930956e+49, 3.298379423780387e+49]
11
first: [6.94056, -10.2813, -9.42732, 9.87348, 6.85065, -8.09456]
retest: [1.05097505493492e+49, -2.6965303151936345e+49, 2.9490282812722873e+49, 9.273789608704201e+49, -3.3924191177983075e+49, 3.0166592768270066e+49]
14
first: [-1.00916e+49, 2.42702e+48, -1.29311e+49, 8.88328e+48, 4.03457e+49, 1.59684e+49]
retest: [-1.7530467933239696e+49, 2.142059656089946e+49, 3.849895413706176e+48, 3.6637271498412496e+49, -1.0804686709474038e+49, 3.847324787480948e+49]
15
first: [3.21188e+49, 9.07834e+48, -3.1699e+49,

In [25]:
print(len(final_wrong))

23


In [26]:
import numpy as np
np.savetxt("output3_labels2_60", output, delimiter=",")
np.savetxt("execution_time3_labels2_60", output, delimiter=",")